# Modelos avanzados y optimización de hiperparámetros

## Objetivo

Ampliar el baseline de churn mediante la comparación de regresión logística, Random Forest y XGBoost. Optuna ajustará XGBoost para maximizar el F1 de la clase `Yes`; los resultados se evaluarán en el conjunto de prueba y se registrarán en MLflow.

## 1. Datos preparados

Se parte del conjunto preparado en la etapa anterior. En esta ejecución se verifican sus dimensiones, la distribución de `Churn` y los valores faltantes de `TotalCharges`.

In [8]:
import pandas as pd
from pathlib import Path

ruta_datos = Path("data/processed/telco_churn_preparado.csv")
df = pd.read_csv(ruta_datos)

print(f"Dimensiones: {df.shape}")
print("\nDistribución de Churn:")
display(df["Churn"].value_counts().rename_axis("Churn").to_frame("Cantidad"))
print(f"\nValores faltantes en TotalCharges: {df['TotalCharges'].isna().sum()}")

Dimensiones: (7043, 21)

Distribución de Churn:


,Cantidad
Churn,
No,5174
Yes,1869



Valores faltantes en TotalCharges: 11


## 2. Preparación para entrenamiento y prueba

Se excluye `customerID` de los predictores y se codifica `Churn` como variable binaria (`No` = 0, `Yes` = 1). Se reserva el 20 % para la evaluación final; la partición estratificada mantiene la proporción de clases y usa `random_state=31` para reproducibilidad.

In [9]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["customerID", "Churn"]).copy()
columnas_enteras = X.select_dtypes(include=["int"]).columns
X[columnas_enteras] = X[columnas_enteras].astype("float64")
y = df["Churn"].map({"No": 0, "Yes": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=31,
    stratify=y,
)

print(f"Entrenamiento: X={X_train.shape}, y={y_train.shape}")
print(f"Prueba: X={X_test.shape}, y={y_test.shape}")
print("\nProporción de Churn en entrenamiento (%):")
display(y_train.value_counts(normalize=True).mul(100).round(3))
print("\nProporción de Churn en prueba (%):")
display(y_test.value_counts(normalize=True).mul(100).round(3))

Entrenamiento: X=(5634, 19), y=(5634,)
Prueba: X=(1409, 19), y=(1409,)

Proporción de Churn en entrenamiento (%):


Churn
0    73.465
1    26.535
Name: proportion, dtype: float64


Proporción de Churn en prueba (%):


Churn
0    73.456
1    26.544
Name: proportion, dtype: float64

## 3. Preprocesamiento para la comparación

Se definió un preprocesamiento común dentro de cada pipeline. Las variables numéricas se imputan con la mediana y se escalan; las categóricas se imputan con la categoría más frecuente y se codifican con One-Hot Encoding. Al incorporarlo a cada modelo, los transformadores se ajustan solo con los datos de entrenamiento, incluso durante la validación cruzada de Optuna.

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

var_num = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
var_cat = X_train.select_dtypes(include=["str", "object", "category"]).columns.tolist()

pipeline_num = Pipeline(steps=[
    ("imputar_num", SimpleImputer(strategy="median")),
    ("escalar", StandardScaler()),
])

pipeline_cat = Pipeline(steps=[
    ("imputar_cat", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocesador = ColumnTransformer(transformers=[
    ("num", pipeline_num, var_num),
    ("cat", pipeline_cat, var_cat),
])

print(f"Variables numéricas: {len(var_num)}")
print(f"Variables categóricas: {len(var_cat)}")

Variables numéricas: 4
Variables categóricas: 15


## 4. Configuración de MLflow

Las comparaciones, pruebas de Optuna y evaluaciones finales se registran en el experimento `telco-churn-advanced` del servidor local.

In [11]:
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
MLFLOW_EXPERIMENT_NAME = "telco-churn-advanced"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
experimento = mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
cliente_mlflow = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experimento: {experimento.name} (ID: {experimento.experiment_id})")

Tracking URI: http://127.0.0.1:5000
Experimento: telco-churn-advanced (ID: 2)


c:\Users\user\Desktop\ESPECIALIZACION_SEMESTRE_2\PROYECTO\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 5. Comparación de modelos

Se comparan regresión logística, Random Forest y XGBoost con validación cruzada estratificada de tres particiones sobre entrenamiento. Se registran accuracy, precision, recall, F1 de `Yes` y ROC-AUC; F1 de `Yes` es la métrica principal.

In [12]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, make_scorer, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from xgboost import XGBClassifier

semilla = 31
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=semilla)
scorer_f1_yes = make_scorer(f1_score, pos_label=1, zero_division=0)

modelos_base = {
    "Regresión logística": LogisticRegression(max_iter=1000, random_state=semilla),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=semilla, n_jobs=1),
    "XGBoost": XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        objective="binary:logistic", eval_metric="logloss", tree_method="hist",
        random_state=semilla, n_jobs=1,
    ),
}
scoring = {
    "accuracy": "accuracy",
    "precision_yes": make_scorer(precision_score, pos_label=1, zero_division=0),
    "recall_yes": make_scorer(recall_score, pos_label=1, zero_division=0),
    "f1_yes": scorer_f1_yes,
    "roc_auc": "roc_auc",
}

resultados_cv = []
for nombre, estimador in modelos_base.items():
    pipeline_modelo = Pipeline(steps=[("preprocesador", clone(preprocesador)), ("modelo", estimador)])
    with mlflow.start_run(run_name=f"cv_{nombre.lower().replace(' ', '_')}"):
        puntajes = cross_validate(
            pipeline_modelo, X_train, y_train, scoring=scoring, cv=cv,
            n_jobs=1, error_score="raise",
        )
        fila = {"Modelo": nombre}
        for metrica in scoring:
            valores = puntajes[f"test_{metrica}"]
            fila[f"{metrica}_media"] = float(valores.mean())
            fila[f"{metrica}_desviacion"] = float(valores.std())
            mlflow.log_metric(f"cv_{metrica}_media", fila[f"{metrica}_media"])
            mlflow.log_metric(f"cv_{metrica}_desviacion", fila[f"{metrica}_desviacion"])
        mlflow.log_params({"modelo": nombre, "cv_folds": 3, "random_state": semilla})
        mlflow.set_tags({"etapa": "comparacion_cv", "dataset": "telco-churn"})
        resultados_cv.append(fila)

tabla_cv = pd.DataFrame(resultados_cv).sort_values("f1_yes_media", ascending=False).reset_index(drop=True)
display(tabla_cv.round(3))

🏃 View run cv_regresión_logística at: http://127.0.0.1:5000/#/experiments/2/runs/52e9e529111b426b9ba86a5d5b8f82d6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
🏃 View run cv_random_forest at: http://127.0.0.1:5000/#/experiments/2/runs/73d409fad2f741119b90758941379f9b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
🏃 View run cv_xgboost at: http://127.0.0.1:5000/#/experiments/2/runs/0679bd528a26443f88e934038df08965
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


,Modelo,accuracy_media,accuracy_desviacion,precision_yes_media,precision_yes_desviacion,recall_yes_media,recall_yes_desviacion,f1_yes_media,f1_yes_desviacion,roc_auc_media,roc_auc_desviacion
0,Regresión logística,0.805,0.003,0.658,0.010,0.557,0.020,0.603,0.011,0.848,0.010
1,XGBoost,0.789,0.009,0.620,0.022,0.528,0.010,0.570,0.014,0.827,0.013
2,Random Forest,0.794,0.007,0.642,0.018,0.507,0.013,0.566,0.013,0.828,0.012


## 6. Optimización de XGBoost con Optuna

Optuna maximiza el F1 de la clase `Yes` mediante validación cruzada sobre entrenamiento. Se ejecutan 20 pruebas y cada prueba registra sus hiperparámetros y resultado como run anidado de MLflow. El conjunto de prueba no participa en la búsqueda.

In [13]:
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import cross_val_score

N_TRIALS = 20

def objetivo_xgboost(trial):
    parametros = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400, step=100),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 4.0),
    }
    pipeline_trial = Pipeline(steps=[
        ("preprocesador", clone(preprocesador)),
        ("modelo", XGBClassifier(
            objective="binary:logistic", eval_metric="logloss", tree_method="hist",
            random_state=semilla, n_jobs=1, **parametros,
        )),
    ])
    with mlflow.start_run(nested=True, run_name=f"optuna_trial_{trial.number:02d}"):
        puntajes = cross_val_score(
            pipeline_trial, X_train, y_train, scoring=scorer_f1_yes, cv=cv,
            n_jobs=1, error_score="raise",
        )
        f1_medio = float(puntajes.mean())
        mlflow.log_params(parametros)
        mlflow.log_metric("cv_f1_yes_media", f1_medio)
        mlflow.log_metric("cv_f1_yes_desviacion", float(puntajes.std()))
        mlflow.set_tags({"etapa": "optuna", "dataset": "telco-churn", "metrica_objetivo": "f1_yes"})
    return f1_medio

with mlflow.start_run(run_name="optuna_xgboost_f1_yes") as run_optuna:
    mlflow.log_params({"modelo": "XGBClassifier", "n_trials": N_TRIALS, "cv_folds": 3, "sampler": "TPESampler", "seed": semilla})
    mlflow.set_tags({"etapa": "optimizacion_hiperparametros", "dataset": "telco-churn"})
    estudio = optuna.create_study(
        direction="maximize", study_name="telco-churn-xgboost-f1",
        sampler=TPESampler(seed=semilla),
    )
    estudio.optimize(objetivo_xgboost, n_trials=N_TRIALS, show_progress_bar=True)
    mlflow.log_metric("best_cv_f1_yes", float(estudio.best_value))
    mlflow.log_params({f"best_{k}": v for k, v in estudio.best_params.items()})
    run_id_optuna = run_optuna.info.run_id

tabla_optuna = estudio.trials_dataframe(attrs=("number", "value", "params", "state"))
display(tabla_optuna.sort_values("value", ascending=False).head(10).round(4))
print(f"Mejor F1 de validación cruzada: {estudio.best_value:.4f}")
print(f"Mejores hiperparámetros: {estudio.best_params}")

[I 2026-10-11 18:25:52,456] A new study created in memory with name: telco-churn-xgboost-f1
Best trial: 0. Best value: 0.592296:   5%|▌         | 1/20 [00:02<00:46,  2.46s/it]

🏃 View run optuna_trial_00 at: http://127.0.0.1:5000/#/experiments/2/runs/e04e5f63ab374b539bbddf54c9db109b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-10-11 18:25:54,894] Trial 0 finished with value: 0.5922964565258863 and parameters: {'n_estimators': 200, 'max_depth': 8, 'learning_rate': 0.13735535793243042, 'min_child_weight': 9.881830027832166, 'subsample': 0.6832661847618885, 'colsample_bytree': 0.6547668195375312, 'reg_alpha': 4.541869011449501, 'reg_lambda': 0.3431925898856014, 'scale_pos_weight': 1.2259981670189715}. Best is trial 0 with value: 0.5922964565258863.


Best trial: 1. Best value: 0.62656:  10%|█         | 2/20 [00:10<01:38,  5.47s/it] 

🏃 View run optuna_trial_01 at: http://127.0.0.1:5000/#/experiments/2/runs/438d750b025a48c683c45a2133b2e4c1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-10-11 18:26:02,463] Trial 1 finished with value: 0.6265603036881098 and parameters: {'n_estimators': 300, 'max_depth': 2, 'learning_rate': 0.03671480476359432, 'min_child_weight': 7.017043957130068, 'subsample': 0.7716676090247373, 'colsample_bytree': 0.6175825229945613, 'reg_alpha': 0.9714299437451029, 'reg_lambda': 2.2332974198638538, 'scale_pos_weight': 1.1877198343060316}. Best is trial 1 with value: 0.6265603036881098.


Best trial: 2. Best value: 0.62687:  15%|█▌        | 3/20 [00:18<01:56,  6.86s/it]

...

🏃 View run optuna_trial_18 at: http://127.0.0.1:5000/#/experiments/2/runs/a51e9b694a0046b3bde2a221d9222927
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-10-11 18:27:22,396] Trial 18 finished with value: 0.6388126744976342 and parameters: {'n_estimators': 400, 'max_depth': 3, 'learning_rate': 0.03133973728918808, 'min_child_weight': 3.749452340170525, 'subsample': 0.8037338654355843, 'colsample_bytree': 0.6342246999276333, 'reg_alpha': 1.0848001951391242, 'reg_lambda': 1.3848375204679195, 'scale_pos_weight': 2.1751545277797066}. Best is trial 14 with value: 0.6407049514991248.


Best trial: 14. Best value: 0.640705: 100%|██████████| 20/20 [01:31<00:00,  4.59s/it]

🏃 View run optuna_trial_19 at: http://127.0.0.1:5000/#/experiments/2/runs/3d233d256f4645fe85d4e3ca014f8abc
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[I 2026-10-11 18:27:24,237] Trial 19 finished with value: 0.6370361959201968 and parameters: {'n_estimators': 400, 'max_depth': 3, 'learning_rate': 0.018791415773187627, 'min_child_weight': 5.2882096060237025, 'subsample': 0.8601158467087422, 'colsample_bytree': 0.7852369310569338, 'reg_alpha': 1.3301022168498595, 'reg_lambda': 1.1028339991098632, 'scale_pos_weight': 1.817506951617924}. Best is trial 14 with value: 0.6407049514991248.
🏃 View run optuna_xgboost_f1_yes at: http://127.0.0.1:5000/#/experiments/2/runs/a16464c609f748f692dd88dc46e3f639
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


,number,value,params_colsample_bytree,params_learning_rate,params_max_depth,params_min_child_weight,params_n_estimators,params_reg_alpha,params_reg_lambda,params_scale_pos_weight,params_subsample,state
14,14,0.6407,0.7397,0.0303,3,3.7401,400,0.3379,0.1992,1.8233,0.8210,COMPLETE
17,17,0.6394,0.6346,0.0298,3,1.8699,400,1.9169,0.8547,1.8157,0.8694,COMPLETE
18,18,0.6388,0.6342,0.0313,3,3.7495,400,1.0848,1.3848,2.1752,0.8037,COMPLETE
10,10,0.6373,0.6265,0.1071,2,1.4893,400,0.6489,0.8757,2.3144,0.8446,COMPLETE
16,16,0.6371,0.7554,0.1273,2,3.0095,400,0.8128,1.5639,2.2640,0.7310,COMPLETE
19,19,0.6370,0.7852,0.0188,3,5.2882,400,1.3301,1.1028,1.8175,0.8601,COMPLETE
5,5,0.6362,0.7215,0.0132,3,2.3429,400,1.5088,0.0369,2.7384,0.7979,COMPLETE
3,3,0.6358,0.7846,0.0228,3,7.2141,400,3.3761,0.4287,1.7020,0.7536,COMPLETE
13,13,0.6353,0.6091,0.0599,2,2.3590,300,2.2036,0.2722,2.9369,0.7955,COMPLETE
4,4,0.6342,0.9644,0.0136,2,1.7557,300,4.1600,3.7504,2.4078,0.7291,COMPLETE


Mejor F1 de validación cruzada: 0.6407
Mejores hiperparámetros: {'n_estimators': 400, 'max_depth': 3, 'learning_rate': 0.030266955384123266, 'min_child_weight': 3.740097212594078, 'subsample': 0.8209828647284638, 'colsample_bytree': 0.7396982945479587, 'reg_alpha': 0.3378963995867015, 'reg_lambda': 0.19915729678654281, 'scale_pos_weight': 1.8233142074658102}


## 7. Evaluación final en el conjunto de prueba

Los modelos de referencia y el XGBoost optimizado se ajustan con todo el conjunto de entrenamiento y se evalúan en el test reservado. El candidato se selecciona por F1 promedio de validación cruzada, no según los resultados de prueba.

In [14]:
from sklearn.metrics import accuracy_score, roc_auc_score

modelos_finales = dict(modelos_base)
modelos_finales["XGBoost optimizado"] = XGBClassifier(
    objective="binary:logistic", eval_metric="logloss", tree_method="hist",
    random_state=semilla, n_jobs=1, **estudio.best_params,
)
modelos_ajustados = {}
resultados_test = []
for nombre, estimador in modelos_finales.items():
    modelo = Pipeline(steps=[("preprocesador", clone(preprocesador)), ("modelo", estimador)])
    modelo.fit(X_train, y_train)
    predicciones = modelo.predict(X_test)
    probabilidades_yes = modelo.predict_proba(X_test)[:, 1]
    metricas = {
        "accuracy": float(accuracy_score(y_test, predicciones)),
        "precision_yes": float(precision_score(y_test, predicciones, pos_label=1, zero_division=0)),
        "recall_yes": float(recall_score(y_test, predicciones, pos_label=1, zero_division=0)),
        "f1_yes": float(f1_score(y_test, predicciones, pos_label=1, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_test, probabilidades_yes)),
    }
    modelos_ajustados[nombre] = modelo
    resultados_test.append({"Modelo": nombre, **metricas})

tabla_test = pd.DataFrame(resultados_test).sort_values("f1_yes", ascending=False).reset_index(drop=True)
f1_cv_por_modelo = {fila["Modelo"]: fila["f1_yes_media"] for fila in resultados_cv}
f1_cv_por_modelo["XGBoost optimizado"] = float(estudio.best_value)
mejor_modelo_cv = max(f1_cv_por_modelo, key=f1_cv_por_modelo.get)
display(tabla_test.round(3))
print(f"Candidato seleccionado por CV: {mejor_modelo_cv}")
print(f"F1 medio de CV: {f1_cv_por_modelo[mejor_modelo_cv]:.3f}")

,Modelo,accuracy,precision_yes,recall_yes,f1_yes,roc_auc
0,XGBoost optimizado,0.777,0.569,0.660,0.611,0.830
1,Regresión logística,0.784,0.613,0.500,0.551,0.826
2,XGBoost,0.764,0.565,0.487,0.523,0.807
3,Random Forest,0.774,0.595,0.460,0.519,0.813


Candidato seleccionado por CV: XGBoost optimizado
F1 medio de CV: 0.641


## 8. Registro de resultados y modelos finales en MLflow

Las métricas finales, parámetros y pipelines se almacenan en runs separados. Para `skops` se autorizan tipos concretos observados en los modelos creados en este experimento.

In [15]:
runs_modelos_finales = {}
for nombre, modelo in modelos_ajustados.items():
    fila = tabla_test.loc[tabla_test["Modelo"] == nombre].iloc[0]
    metricas_log = {f"test_{col}": float(fila[col]) for col in ("accuracy", "precision_yes", "recall_yes", "f1_yes", "roc_auc")}
    estimador = modelo.named_steps["modelo"]
    if nombre == "Random Forest":
        tipos_confiables = ["numpy.dtype", "sklearn.tree._tree.Tree"]
    elif nombre.startswith("XGBoost"):
        tipos_confiables = ["numpy.dtype", "xgboost.core.Booster", "xgboost.sklearn.XGBClassifier"]
    else:
        tipos_confiables = ["numpy.dtype"]

    parametros_modelo = {
        clave: valor for clave, valor in estimador.get_params().items()
        if clave in {"max_iter", "n_estimators", "max_depth", "learning_rate", "min_child_weight", "subsample", "colsample_bytree", "reg_alpha", "reg_lambda", "scale_pos_weight"}
        and valor is not None
    }
    with mlflow.start_run(run_name=f"final_{nombre.lower().replace(' ', '_')}") as run:
        mlflow.log_params({"modelo": nombre, "random_state": semilla, **parametros_modelo})
        mlflow.log_metrics(metricas_log)
        mlflow.log_metric("cv_f1_yes_media", float(f1_cv_por_modelo[nombre]))
        mlflow.set_tags({
            "etapa": "evaluacion_final", "dataset": "telco-churn",
            "seleccionado_por_cv": str(nombre == mejor_modelo_cv).lower(),
        })
        mlflow.sklearn.log_model(
            sk_model=modelo, name="modelo", input_example=X_test.head(3),
            skops_trusted_types=tipos_confiables,
        )
        runs_modelos_finales[nombre] = run.info.run_id

with mlflow.start_run(run_name="final_model_comparison"):
    tabla_registro = tabla_test.copy()
    tabla_registro["f1_yes_cv"] = tabla_registro["Modelo"].map(f1_cv_por_modelo)
    tabla_registro["seleccionado_por_cv"] = tabla_registro["Modelo"].eq(mejor_modelo_cv)
    mlflow.log_table(tabla_registro, artifact_file="comparacion_modelos_test.json")

display(pd.DataFrame([{"Modelo": nombre, "run_id": run_id} for nombre, run_id in runs_modelos_finales.items()]))

c:\Users\user\Desktop\ESPECIALIZACION_SEMESTRE_2\PROYECTO\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run final_regresión_logística at: http://127.0.0.1:5000/#/experiments/2/runs/43e162bf56674ca5a4ca1fb7cb0dcfb1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


c:\Users\user\Desktop\ESPECIALIZACION_SEMESTRE_2\PROYECTO\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run final_random_forest at: http://127.0.0.1:5000/#/experiments/2/runs/244dbea8eccf49f5a34baaeb8d0da9ea
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


c:\Users\user\Desktop\ESPECIALIZACION_SEMESTRE_2\PROYECTO\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run final_xgboost at: http://127.0.0.1:5000/#/experiments/2/runs/23b13d5a72bf46619587b30428178929
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


c:\Users\user\Desktop\ESPECIALIZACION_SEMESTRE_2\PROYECTO\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run final_xgboost_optimizado at: http://127.0.0.1:5000/#/experiments/2/runs/cb2ff2894c2f40189eddfb2222cc48be
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


KeyboardInterrupt: 

## 9. Verificación del modelo seleccionado

Se carga desde MLflow el candidato seleccionado por validación cruzada y se compara su salida con la del pipeline ajustado en cinco registros de prueba.

In [ ]:
run_id_seleccionado = runs_modelos_finales[mejor_modelo_cv]
modelo_cargado = mlflow.pyfunc.load_model(f"runs:/{run_id_seleccionado}/modelo")
predicciones_mlflow = modelo_cargado.predict(X_test.head(5))
predicciones_directas = modelos_ajustados[mejor_modelo_cv].predict(X_test.head(5))
verificacion = pd.DataFrame({
    "real": y_test.head(5).to_numpy(),
    "prediccion_pipeline": predicciones_directas,
    "prediccion_mlflow": predicciones_mlflow,
})
verificacion["coinciden"] = verificacion["prediccion_pipeline"] == verificacion["prediccion_mlflow"]
display(verificacion)
assert verificacion["coinciden"].all(), "Las predicciones del modelo cargado no coinciden."
print(f"Modelo MLflow cargado y verificado: {mejor_modelo_cv}")

,real,prediccion_pipeline,prediccion_mlflow,coinciden
0,0,0,0,True
1,1,1,1,True
2,0,0,0,True
3,0,0,0,True
4,0,0,0,True


Modelo MLflow cargado y verificado: XGBoost optimizado


## 10. Conclusiones

La validación cruzada compara los modelos de referencia y guía la selección del candidato; Optuna ajusta XGBoost sin utilizar el conjunto de prueba. La tabla de test presenta la evaluación final de cada modelo y MLflow conserva parámetros, métricas, runs de Optuna y artefactos para reproducir y revisar los resultados.

In [ ]:
f1_logistica_cv = f1_cv_por_modelo["Regresión logística"]
f1_xgboost_base_cv = f1_cv_por_modelo["XGBoost"]
f1_xgboost_opt_cv = f1_cv_por_modelo["XGBoost optimizado"]
f1_logistica_test = float(tabla_test.loc[tabla_test["Modelo"] == "Regresión logística", "f1_yes"].iloc[0])
f1_xgboost_opt_test = float(tabla_test.loc[tabla_test["Modelo"] == "XGBoost optimizado", "f1_yes"].iloc[0])

print(f"Mejor candidato por F1 medio de CV: {mejor_modelo_cv} ({f1_cv_por_modelo[mejor_modelo_cv]:.3f}).")
print(f"Optuna cambió el F1 medio de XGBoost de {f1_xgboost_base_cv:.3f} a {f1_xgboost_opt_cv:.3f} en CV.")
print(f"En el test reservado, el F1 de XGBoost optimizado fue {f1_xgboost_opt_test:.3f}; el de regresión logística fue {f1_logistica_test:.3f}.")
print("El conjunto de prueba se usó únicamente para esta evaluación final; la selección se hizo con validación cruzada.")

Mejor candidato por F1 medio de CV: XGBoost optimizado (0.641).
Optuna cambió el F1 medio de XGBoost de 0.570 a 0.641 en CV.
En el test reservado, el F1 de XGBoost optimizado fue 0.611; el de regresión logística fue 0.551.
El conjunto de prueba se usó únicamente para esta evaluación final; la selección se hizo con validación cruzada.
